In [1]:
using Pkg
using SpeedyWeather, GLMakie, Statistics
using SpeedyWeatherInternals.Utils
using LowerTriangularArrays
using SpeedyTransforms

In [2]:
include("TrenberthCallbacks.jl")  # loads module into Main
using .TrenberthCallbacks

In [3]:
# spectral_grid = SpectralGrid(trunc=31, nlayers=8)
# model = PrimitiveWetModel(spectral_grid)

In [4]:
# using SpeedyWeather.Radiation
# using SpeedyWeather
# subtypes(AbstractShortwave)

spectral_grid = SpectralGrid(trunc=31, nlayers=8)
# model = PrimitiveWetModel(spectral_grid; shortwave_radiation=OneBandShortwave(spectral_grid))
model = PrimitiveWetModel(spectral_grid; shortwave_radiation=OneBandShortwave(spectral_grid, clouds = DiagnosticClouds(spectral_grid; use_stratocumulus=true)))


# # get surface shortwave radiation down
# ssrd = simulation.diagnostic_variables.physics.surface_shortwave_down
# heatmap(ssrd,title="Surface shortwave radiation down [W/m^2]")

PrimitiveWetModel <: PrimitiveWet
├ spectral_grid: SpectralGrid{CPU{KernelAbstractions.CPU}, Spectrum{CPU{KernelAbstractions.CPU}...
├ architecture: CPU{KernelAbstractions.CPU}
├ dynamics: Bool
├ geometry: Geometry{SpectralGrid{CPU{KernelAbstractions.CPU}, Spectrum{CPU{KernelAbstractions....
├ planet: Earth{Float32}
├ atmosphere: EarthAtmosphere{Float32}
├ coriolis: Coriolis{Vector{Float32}}
├ geopotential: Geopotential{Vector{Float32}}
├ adiabatic_conversion: AdiabaticConversion{Vector{Float32}}
├ particle_advection: Nothing
├ initial_conditions: InitialConditions{ZonalWind{Float32}, PressureOnOrography, JablonowskiTem...
├ forcing: Nothing
├ drag: Nothing
├ random_process: Nothing
├ tracers: Dict{Symbol, Tracer}
├ orography: EarthOrography{Float32, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{...
├ land_sea_mask: EarthLandSeaMask{Float32, Field{Float32, 1, Vector{Float32}, OctahedralGaussia...
├ ocean: SlabOcean{Float32}
├ sea_ice: ThermodynamicSeaIce{Float32}
├ land: La

In [5]:
# simulation = initialize!(model)

In [6]:
# using SpeedyWeatherInternals
# run!(simulation, period=Week(1))

In [7]:
# spectral_grid = SpectralGrid()

# deciding between the cloud schemes:
# use without stratocumulus clouds
# sw_no_sc = OneBandShortwave(spectral_grid, clouds = DiagnosticClouds(spectral_grid; use_stratocumulus=false))

# use with stratocumulus clouds
# sw_with_sc = OneBandShortwave(spectral_grid, clouds = DiagnosticClouds(spectral_grid; use_stratocumulus=true))

## Output variables

In [8]:
# see the model output structure:
model.output

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ humid: specific humidity [kg/kg]
 ├ temp: temperature [degC]
 ├ u: zonal wind [m/s]
 ├ mslp: mean sea-level pressure [hPa]
 └ vor: relative vorticity [s^-1]

In [9]:
add!(model, SpeedyWeather.RadiationOutput()...) # add radiation diagnostics to model output
SpeedyWeather.RadiationOutput()


(SpeedyWeather.OutgoingLongwaveRadiationOutput <: SpeedyWeather.AbstractOutputVariable
├ name::String = olr
├ unit::String = W/m^2
├ long_name::String = Outgoing longwave radiation
├ dims_xyzt::NTuple{4, Bool} = (true, true, false, true)
├ missing_value::Float64 = NaN
├ compression_level::Int64 = 1
├ shuffle::Bool = false
├ keepbits::Int64 = 7, SpeedyWeather.OutgoingShortwaveRadiationOutput <: SpeedyWeather.AbstractOutputVariable
├ name::String = osr
├ unit::String = W/m^2
├ long_name::String = Outgoing shortwave radiation
├ dims_xyzt::NTuple{4, Bool} = (true, true, false, true)
├ missing_value::Float64 = NaN
├ compression_level::Int64 = 1
├ shuffle::Bool = false
├ keepbits::Int64 = 7, SpeedyWeather.SurfaceShortwaveUpOutput <: SpeedyWeather.AbstractOutputVariable
├ name::String = sru
├ unit::String = W/m^2
├ long_name::String = Surface shortwave radiation up
├ dims_xyzt::NTuple{4, Bool} = (true, true, false, true)
├ missing_value::Float64 = NaN
├ compression_level::Int64 = 1
├ shuffle:

In [10]:
add!(model, SpeedyWeather.SurfaceFluxesOutput()...) # add surface flux diagnostics to model output

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ shuf: Surface humidity flux (positive up) [kg/s/m^2]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ shf: Surface sensible heat flux (p

#### adding callbacks: 

In [11]:
# --------------------------
# helper: compute Trenberth diagnostics from diagn + model
# --------------------------
# - calc_trenberth_from_diagn
function calc_trenberth_from_diagn(diagn, model; SumFlag::Bool=false)
    fields = Dict(
        # :LHF   => diagn.physics.surface_latent_heat_flux,
        :LHF   => diagn.physics.surface_humidity_flux,
        :SHF   => diagn.physics.sensible_heat_flux,
        :SSRU  => diagn.physics.surface_shortwave_up,
        :SLRU  => diagn.physics.surface_longwave_up,
        :SSRD  => diagn.physics.surface_shortwave_down,
        :SLRD  => diagn.physics.surface_longwave_down,
        :OSR   => diagn.physics.outgoing_shortwave,
        :OLR   => diagn.physics.outgoing_longwave,
        :albedo=> diagn.physics.albedo
    )

    function calc_global_mean(field)
        a = transform(field)
        a00 = real(a[1])
        return a00 / model.spectral_transform.norm_sphere
    end

    function calc_global_sum(field)
        mean_val = calc_global_mean(field)
        area = 4π * model.planet.radius^2
        return mean_val * area
    end

    calcfun = SumFlag ? calc_global_sum : calc_global_mean

    results = Dict{Symbol, Float64}()
    for (k, f) in fields
        try
        if k == :LHF
            results[k] = calcfun(f .* 2.5e6)  # convert moisture flux to latent heat flux (W/m² or W)
        else
            results[k] = calcfun(f)
        end
        catch err
            @warn "calc_trenberth_from_diagn: could not compute $k: $err"
            results[k] = NaN
        end
    end

    results[:SW_net_sfc]  = results[:SSRD] - results[:SSRU]
    results[:LW_net_sfc]  = results[:SLRD] - results[:SLRU]
    results[:surface_net] = results[:SW_net_sfc] + results[:LW_net_sfc] - results[:LHF] - results[:SHF]

    return results
end


calc_trenberth_from_diagn (generic function with 1 method)

In [12]:
# Default long names for Trenberth variables
const TRENBERTH_LONGNAMES = Dict(
    :LHF => "Surface latent heat flux (W/m²)",
    :SHF => "Surface sensible heat flux (W/m²)",
    :SSRU => "Surface shortwave up (W/m²)",
    :SLRU => "Surface longwave up (W/m²)",
    :SSRD => "Surface shortwave down (W/m²)",
    :SLRD => "Surface longwave down (W/m²)",
    :OSR => "Outgoing shortwave radiation (TOA) (W/m²)",
    :OLR => "Outgoing longwave radiation (TOA) (W/m²)",
    :albedo => "Surface albedo",
    :SW_net_sfc => "Surface net shortwave (W/m²)",
    :LW_net_sfc => "Surface net longwave (W/m²)",
    :surface_net => "Surface net energy (W/m²)"
)

Dict{Symbol, String} with 12 entries:
  :surface_net => "Surface net energy (W/m²)"
  :SLRU        => "Surface longwave up (W/m²)"
  :LHF         => "Surface latent heat flux (W/m²)"
  :LW_net_sfc  => "Surface net longwave (W/m²)"
  :OLR         => "Outgoing longwave radiation (TOA) (W/m²)"
  :albedo      => "Surface albedo"
  :SSRD        => "Surface shortwave down (W/m²)"
  :SW_net_sfc  => "Surface net shortwave (W/m²)"
  :OSR         => "Outgoing shortwave radiation (TOA) (W/m²)"
  :SLRD        => "Surface longwave down (W/m²)"
  :SHF         => "Surface sensible heat flux (W/m²)"
  :SSRU        => "Surface shortwave up (W/m²)"

In [13]:
using Dates

# Convert various time types to Float64. Default unit = :seconds.
function time_to_float(t; unit::Symbol = :seconds)
    if t isa DateTime
        secs = Dates.datetime2unix(t)                     # seconds since Unix epoch
        return unit == :seconds ? Float64(secs) :
               unit == :days    ? Float64(secs / 86400.0) :
               error("unsupported unit: $unit")
    elseif t <: Dates.Period   # Day, Hour, Minute, etc.
        # Dates.value returns the integer magnitude in the Period's base units
        # For Day it returns number of days, for Hour number of hours, etc.
        # Convert to days or seconds depending on unit
        if unit == :days
            return float(Dates.value(t))
        elseif unit == :seconds
            # approximate: convert days/hours etc. to seconds using common ratios
            # We'll convert via Day/Hr/Minute explicitly for safety:
            if t isa Day
                return float(Dates.value(t) * 86400)
            elseif t isa Hour
                return float(Dates.value(t) * 3600)
            elseif t isa Minute
                return float(Dates.value(t) * 60)
            else
                # fallback: convert to days then seconds
                return float(Dates.value(Day(round(Int, Dates.value(t)))) * 86400)
            end
        else
            error("unsupported unit: $unit")
        end
    elseif t isa Number
        return float(t)
    else
        error("unsupported time type: $(typeof(t))")
    end
end


time_to_float (generic function with 1 method)

#### setting the callback and its schedule:

In [14]:
Base.@kwdef mutable struct TrenberthCallback <: SpeedyWeather.AbstractCallback
    timestep_counter::Int = 0
    data::Dict{Symbol, Vector{Float64}} = Dict{Symbol, Vector{Float64}}()
    times::Vector{Float64} = Float64[]          # elapsed seconds
    datetimes::Vector{DateTime} = DateTime[]    # original DateTime stamps
    start_time::Float64 = 0.0
    SumFlag::Bool = false
    var_longnames::Dict{Symbol,String} = TRENBERTH_LONGNAMES
    schedule::Schedule = Schedule()  # default: runs every timestep
end

# Constructor function to create instances with smart allocation
function TrenberthCallback(; vars = [:LHF,:SHF,:SSRU,:SLRU,:SSRD,:SLRD,:OSR,:OLR,:albedo,:SW_net_sfc,:LW_net_sfc,:surface_net],
                             SumFlag::Bool=false,
                             nsteps::Int=0,
                             var_longnames::Dict{Symbol,String}=TRENBERTH_LONGNAMES,
                             schedule::Schedule=Schedule())
    d = Dict{Symbol, Vector{Float64}}()
    for v in vars
        d[v] = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
    end
    times = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
    datetimes = nsteps > 0 ? Vector{DateTime}(undef, nsteps + 1) : DateTime[]
    return TrenberthCallback(0, d, times, datetimes, 0.0, SumFlag, var_longnames, schedule)
end


TrenberthCallback

In [15]:
# Run every timestep (default)
# cb = TrenberthCallback(SumFlag=false, nsteps=0)

# Run every day
# cb = TrenberthCallback(SumFlag=false, nsteps=0, schedule=Schedule(every=Day(1)))

# Run every 6 hours
# cb = TrenberthCallback(SumFlag=false, nsteps=0, schedule=Schedule(every=Hour(6)))

# Run every 12 hours
cb = TrenberthCallback(SumFlag=false, nsteps=0, schedule=Schedule(every=Hour(12)))

TrenberthCallback <: AbstractCallback
├ timestep_counter::Int64 = 0
├ data::Dict{Symbol, Vector{Float64}} = Dict(:LW_net_sfc => [], :OLR => [], :SSRD => [], :SLRD => [], :SHF => [], :SSRU => [], :SLRU => [], :LHF => [], :albedo => [], :SW_net_sfc => [], :OSR => [], :surface_net => [])
├ start_time::Float64 = 0.0
├ SumFlag::Bool = false
├ var_longnames::Dict{Symbol, String} = Dict(:surface_net => "Surface net energy (W/m²)", :SLRU => "Surface longwave up (W/m²)", :LHF => "Surface latent heat flux (W/m²)", :LW_net_sfc => "Surface net longwave (W/m²)", :OLR => "Outgoing longwave radiation (TOA) (W/m²)", :albedo => "Surface albedo", :SSRD => "Surface shortwave down (W/m²)", :SW_net_sfc => "Surface net shortwave (W/m²)", :OSR => "Outgoing shortwave radiation (TOA) (W/m²)", :SLRD => "Surface longwave down (W/m²)", :SHF => "Surface sensible heat flux (W/m²)", :SSRU => "Surface shortwave up (W/m²)")
├ schedule::Schedule = Schedule <: SpeedyWeather.AbstractSchedule
├ every::Second = 43200 secon

In [16]:
# Pretty-print the long names
function show_var_names(cb::TrenberthCallback)
    for (k, long) in cb.var_longnames
        println(string(k), " → ", long)
    end
    return nothing
end

# assemble a DataFrame if DataFrames.jl is installed
function to_dataframe(cb::TrenberthCallback)
    try
        @eval using DataFrames
    catch
        error("DataFrames.jl not available. Install it with `using Pkg; Pkg.add(\"DataFrames\")`")
    end
    df = DataFrame(time = cb.datetimes)
    for (k, vec) in cb.data
        colname = get(cb.var_longnames, k, string(k))  # column name uses long name if available
        # ensure column identifier is a Symbol
        df[Symbol(colname)] = vec
    end
    return df
end

to_dataframe (generic function with 1 method)

In [17]:
function SpeedyWeather.initialize!(cb::TrenberthCallback,
                                   progn::PrognosticVariables,
                                   diagn::DiagnosticVariables,
                                   model::AbstractModel)
    
    # when initializing a scheduled callback also initialize its schedule!
    initialize!(cb.schedule, progn.clock)

    # Store the simulation start time for reference (convert DateTime to Float64 Unix timestamp)
    cb.start_time = Dates.datetime2unix(progn.clock.time)
    
    # Try to get nsteps, but if it doesn't work, just start with empty vectors
    try
        nsteps = progn.clock.nsteps
        # if our data dict vectors are empty or wrong size, (re)allocate
        for (k, v) in cb.data
            if isempty(v) || length(v) != nsteps + 1
                cb.data[k] = Vector{Float64}(undef, nsteps + 1)
            end
        end
        if isempty(cb.times) || length(cb.times) != nsteps + 1
            cb.times = Vector{Float64}(undef, nsteps + 1)
        end
        if isempty(cb.datetimes) || length(cb.datetimes) != nsteps + 1
            cb.datetimes = Vector{DateTime}(undef, nsteps + 1)
        end
    catch
        # If we can't get nsteps, just use dynamic push mode
        @info "Could not determine nsteps, using dynamic push mode"
    end

    # set counter to 1 and store initial conditions
    cb.timestep_counter = 1
    t0 = Dates.datetime2unix(progn.clock.time)  # Convert DateTime to Unix timestamp
    dt0 = progn.clock.time  # Get the original DateTime object
    # compute initial values using diagn
    res0 = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    for (k, v) in res0
        if haskey(cb.data, k) # check if key exists
            if length(cb.data[k]) > 0
                cb.data[k][1] = v
            else
                push!(cb.data[k], v)
            end
        else
            cb.data[k] = [v]
        end
    end

    # Store time relative to simulation start (in seconds) and DateTime
    if length(cb.times) > 0
        cb.times[1] = t0 - cb.start_time
        cb.datetimes[1] = dt0
    else
        push!(cb.times, t0 - cb.start_time)
        push!(cb.datetimes, dt0)
    end
    return nothing
end

In [18]:
# --- callback! called every step (after the step completes) ---
function SpeedyWeather.callback!(cb::TrenberthCallback,
                                 progn::PrognosticVariables,
                                 diagn::DiagnosticVariables,
                                 model::AbstractModel)
    
    # scheduled callbacks start with this line to execute only when scheduled!
    # else escape immediately
    isscheduled(cb.schedule, progn.clock) || return nothing

    # increment step index
    cb.timestep_counter += 1
    
    # compute current diagnostics
    res = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    
    # push new values to the arrays
    for (k, v) in res
        if !haskey(cb.data, k)
            # new key appeared: create vector and push
            cb.data[k] = [v]
        else
            # existing key: push to the vector
            push!(cb.data[k], v)
        end
    end
    
    # record model time relative to simulation start (in seconds) and DateTime
    # Convert DateTime to Unix timestamp, then subtract start_time to get elapsed seconds
    current_time = Dates.datetime2unix(progn.clock.time)
    push!(cb.times, current_time - cb.start_time)
    push!(cb.datetimes, progn.clock.time)  # Store the DateTime object
    return nothing
end

In [19]:
# --- finalize (optional) ---
using Statistics  # Import mean function

# use the finalize! to clculate the mean values over the entire simulation:
function SpeedyWeather.finalize!(cb::TrenberthCallback,
                                   progn::PrognosticVariables,
                                   diagn::DiagnosticVariables,
                                   model::AbstractModel)
    # compute final diagnostics
    # res = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    # for (k, v) in res
    #     if haskey(cb.data, k)
    #         push!(cb.data[k], v)
    #     else
    #         cb.data[k] = [v]
    #     end
    # end

    # compute the mean over the entire simulation for each variable and print
    # Note: we keep the original data vectors and just print the means
    println("\n=== Simulation Means ===")
    for (k, vec) in cb.data
        mean_val = mean(vec)
        println("Mean $k over simulation: $mean_val")
    end
    return nothing
end

adding the calbak to the model


In [20]:
# A: add into the callbacks dict directly (works if model.callbacks is a Dict-like)
add!(model.callbacks, :trenberth => cb)

In [21]:
keys(model.callbacks)              # should include :trenberth to make sure the callback was added correctly.

KeySet for a Dict{Symbol, SpeedyWeather.AbstractCallback} with 1 entry. Keys:
  :trenberth

In [22]:
model.callbacks[:trenberth] === cb # should be true to make sure the callback was added correctly. 

true

In [23]:
# SpeedyWeather.readable_secs

In [39]:
sim = initialize!(model)   # this will call SpeedyWeather.initialize! on cb
run!(sim, period=Day(20))  # or your usual run invocation

┌ Info: Could not determine nsteps, using dynamic push mode
└ @ Main /home/lucy_h/speedyweather_trenberth_diagram/clouds_joining/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X25sdnNjb2RlLXJlbW90ZQ==.jl:29



=== Simulation Means ===
Mean LW_net_sfc over simulation: -77.40649814293033
Mean OLR over simulation: 244.2517765232774
Mean SSRD over simulation: 268.8333550124872
Mean SLRD over simulation: 335.24087924644596
Mean SHF over simulation: 55.697381504246444
Mean SSRU over simulation: 28.07827715013848
Mean SLRU over simulation: 412.6473773893763
Mean LHF over simulation: 38.97466202429479
Mean albedo over simulation: 0.1261668288316883
Mean SW_net_sfc over simulation: 240.7550778623487
Mean OSR over simulation: 79.41339499051453
Mean surface_net over simulation: 68.67653619087717


In [40]:
cb.data # should contain the Trenberth variables collected during the simulation

Dict{Symbol, Vector{Float64}} with 12 entries:
  :LW_net_sfc  => [0.0, -72.6536, -66.436, -72.9242, -68.0592, -74.3138, -69.60…
  :OLR         => [0.0, 269.602, 268.448, 266.487, 264.436, 262.818, 261.05, 25…
  :SSRD        => [0.0, 261.356, 270.25, 268.445, 270.543, 271.467, 272.187, 27…
  :SLRD        => [0.0, 353.142, 354.526, 354.178, 353.379, 352.648, 351.596, 3…
  :SHF         => [0.0, 64.7518, 45.0865, 56.6862, 43.7429, 56.7295, 43.7139, 5…
  :SSRU        => [0.0, 31.5447, 21.7392, 33.0937, 21.8255, 33.7162, 22.0114, 3…
  :SLRU        => [0.0, 425.796, 420.962, 427.102, 421.438, 426.962, 421.203, 4…
  :LHF         => [0.0, -11.6067, -8.80666, 4.12649, -2.49309, 12.2631, 5.60018…
  :albedo      => [0.0, 0.127152, 0.127229, 0.127333, 0.127438, 0.12754, 0.1276…
  :SW_net_sfc  => [0.0, 229.811, 248.511, 235.352, 248.718, 237.75, 250.176, 23…
  :OSR         => [0.0, 96.8233, 77.773, 90.6341, 77.4719, 87.9748, 75.9084, 85…
  :surface_net => [0.0, 104.012, 145.795, 101.615, 139.409, 94

In [41]:
size(cb.data[:LHF]) # should show the number of time steps + 1 (for initial condition)

(61,)

In [42]:
cb.data[:LHF][:] # should show the last 10 values of the latent heat flux variable

61-element Vector{Float64}:
   0.0
 -11.606696749189831
  -8.806660787507077
   4.126485863215356
  -2.4930897253703863
  12.263147642414811
   5.6001813991636
  21.65711075039676
  14.843475868572275
  31.38783895361904
   ⋮
  62.22089306571456
  71.78467816396935
  65.54149940658581
  75.0227992678635
  67.88353967670702
  75.82042090066395
  68.98204071940499
  76.50033234556317
  69.93815716830274

In [43]:
size(cb.times) # should show the number of time steps + 1 (for initial condition)

(61,)

In [44]:
cb.times

61-element Vector{Float64}:
      0.0
  43200.0
  86400.0
 129600.0
 172800.0
 216000.0
 259200.0
 302400.0
 345600.0
 388800.0
      ⋮
      1.3824e6
      1.4256e6
      1.4688e6
      1.512e6
      1.5552e6
      1.5984e6
      1.6416e6
      1.6848e6
      1.728e6

In [45]:
size(cb.datetimes) # should show the number of time steps + 1 (for initial condition)

(61,)

In [46]:
cb.times

61-element Vector{Float64}:
      0.0
  43200.0
  86400.0
 129600.0
 172800.0
 216000.0
 259200.0
 302400.0
 345600.0
 388800.0
      ⋮
      1.3824e6
      1.4256e6
      1.4688e6
      1.512e6
      1.5552e6
      1.5984e6
      1.6416e6
      1.6848e6
      1.728e6

In [47]:
cb.datetimes # to see the DateTime objects corresponding to the times

61-element Vector{DateTime}:
 2000-01-01T00:00:00
 2000-01-01T12:00:00
 2000-01-02T00:00:00
 2000-01-02T12:00:00
 2000-01-03T00:00:00
 2000-01-03T12:00:00
 2000-01-04T00:00:00
 2000-01-04T12:00:00
 2000-01-05T00:00:00
 2000-01-05T12:00:00
 ⋮
 2000-01-17T00:00:00
 2000-01-17T12:00:00
 2000-01-18T00:00:00
 2000-01-18T12:00:00
 2000-01-19T00:00:00
 2000-01-19T12:00:00
 2000-01-20T00:00:00
 2000-01-20T12:00:00
 2000-01-21T00:00:00

In [48]:
cb.var_longnames

Dict{Symbol, String} with 12 entries:
  :surface_net => "Surface net energy (W/m²)"
  :SLRU        => "Surface longwave up (W/m²)"
  :LHF         => "Surface latent heat flux (W/m²)"
  :LW_net_sfc  => "Surface net longwave (W/m²)"
  :OLR         => "Outgoing longwave radiation (TOA) (W/m²)"
  :albedo      => "Surface albedo"
  :SSRD        => "Surface shortwave down (W/m²)"
  :SW_net_sfc  => "Surface net shortwave (W/m²)"
  :OSR         => "Outgoing shortwave radiation (TOA) (W/m²)"
  :SLRD        => "Surface longwave down (W/m²)"
  :SHF         => "Surface sensible heat flux (W/m²)"
  :SSRU        => "Surface shortwave up (W/m²)"

### Creating observables from callback output

The callback outputs a dictionary with the fluxes as keys. To establish an efficient pipeline with the Makie interface, the dictionary needs to be converted to observables. These are mutable containers that can be listened to and tracked. Makie interacts with these observables and reacts when the observables change. These are ideal for integrating multiple streams of information within one diagram. 

In [49]:
using Observables, Dates

# 1) decide which variables you want in the NamedTuple and in what order
vars_order = [:LHF, :SHF, :SSRU, :SLRU, :SSRD, :SLRD, :OSR, :OLR, :albedo,
              :SW_net_sfc, :LW_net_sfc, :surface_net]
## Lists the variable in time step order. Builds consistent tuples for plotting (
# the plots always find the same fields)

# 2) compute a safe length (use minimum so all fields exist for each index)
lengths = Int[]
for k in vars_order
    if !haskey(cb.data, k)
        error("cb.data missing variable $k")
    end
    push!(lengths, length(cb.data[k]))
end
n = minimum(lengths)            # safe common length; change to maximum if you prefill missing
## cb.data[:LHF] is a vector that the callback filled at runtime. Different vectors may have different
# lengths. Taking the minimum minimises missing entries with NaN or missing. 



# 3) build a Vector of NamedTuples, one per timestep
flux_series = Vector{NamedTuple}(undef, n)
for i in 1:n
    # extract value for each variable at time i (use cb.data[k][i])
    flux_series[i] = (
        datetime = cb.datetimes[i],   # keep the DateTime for convenience
        LHF = cb.data[:LHF][i],
        SHF = cb.data[:SHF][i],
        SSRU = cb.data[:SSRU][i],
        SLRU = cb.data[:SLRU][i],
        SSRD = cb.data[:SSRD][i],
        SLRD = cb.data[:SLRD][i],
        OSR  = cb.data[:OSR][i],
        OLR  = cb.data[:OLR][i],
        albedo = cb.data[:albedo][i],
        SW_net_sfc = cb.data[:SW_net_sfc][i],
        LW_net_sfc = cb.data[:LW_net_sfc][i],
        surface_net = cb.data[:surface_net][i]
    )
end
## Builds a NamedTuple with the same field names. NamedTuple was used because it is convenient and immutables
## Datetime included so current_point also includes the timestamp


# 4) wrap in an Observable for UI binding
history_obs = Observable(flux_series)   # Observable{Vector{NamedTuple}}
## Wraps the whole vector in one observable 
## When history_obs is replaced and observables.notify! is called, derived observables update


# 5) time index observable (for slider)
time_idx = Observable(1)                # integer index, 1..n
## Small observable integer that tracks which timestep is selected


# 6) derived observable for the current timestep (
current_point = map((hist, idx) -> hist[idx], history_obs, time_idx)
# Now current_point[] is the NamedTuple at the selected index.
## Whenever history_obs[] or time_idx[] change, current_point updates automatically


# Example: pull a scalar observable for one flux (e.g., LHF) for easy plotting
current_LHF = map(p -> p.LHF, current_point)   # Observable{Float64} representing current LHF

# Example: print when slider changes
on(current_point) do p
    @info "time = $(p.datetime), LHF = $(p.LHF), SSRD = $(p.SSRD)"
end


ObserverFunction defined at /home/lucy_h/speedyweather_trenberth_diagram/clouds_joining/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X53sdnNjb2RlLXJlbW90ZQ==.jl:69 operating on Observable((datetime = DateTime("2000-01-01T00:00:00"), LHF = 0.0, SHF = 0.0, SSRU = 0.0, SLRU = 0.0, SSRD = 0.0, SLRD = 0.0, OSR = 0.0, OLR = 0.0, albedo = 0.0, SW_net_sfc = 0.0, LW_net_sfc = 0.0, surface_net = 0.0))

### Checking observables 

These observables are scalar. Each observable holds a single numerical value (global mean flux in W/m2) at the currently selected timestep. 

The following line:  map(p -> p.LHF, current point) creates a derived observables from the scalar LHF value from the current_point observable. 

Current point --> holds the flux fields. It is not scalar, but the fields are scalar. 

In [50]:
## Testing observables are working 
println(current_point[])

(datetime = DateTime("2000-01-01T00:00:00"), LHF = 0.0, SHF = 0.0, SSRU = 0.0, SLRU = 0.0, SSRD = 0.0, SLRD = 0.0, OSR = 0.0, OLR = 0.0, albedo = 0.0, SW_net_sfc = 0.0, LW_net_sfc = 0.0, surface_net = 0.0)


### Plotting function

Sets up derived global mean flux observables. 

In [51]:
using GLMakie, Observables, Dates

# Assume flux_series, history_obs, time_idx, current_point are already created
nsteps = length(flux_series)

# Extract individual flux observables 
LHF_obs = map(p -> p.LHF, current_point)
SHF_obs = map(p -> p.SHF, current_point)
SSRU_obs = map(p -> p.SSRU, current_point)
SLRU_obs = map(p -> p.SLRU, current_point)
SSRD_obs = map(p -> p.SSRD, current_point)
SLRD_obs = map(p -> p.SLRD, current_point)
OSR_obs = map(p -> p.OSR, current_point)
OLR_obs = map(p -> p.OLR, current_point)
SW_net_sfc_obs = map(p -> p.SW_net_sfc, current_point)
albedo_obs = map(p -> p.albedo, current_point)
surface_net_obs = map(p -> p.surface_net, current_point)




Observable(0.0)


In [52]:
#Extracting solar constant from model
params = parameters(model)
param_vec = vec(params)
solar_constant = param_vec.planet.solar_constant




1365.0f0

#### Visual setup of figure

In [68]:
# Visual Setup 
fig = Figure(size = (1800, 1000), backgroundcolor = :white, fontsize = 40)
ax = Axis(fig[1, 1], 
    aspect = DataAspect(), 
    backgroundcolor = (:lightblue, 0.12),
    title = "Energy Budget Diagram")

hidedecorations!(ax)
hidespines!(ax)
xlims!(ax, -4.0, 4.0)
ylims!(ax, -2.5, 3.2)

# Arrow drawing 
function draw_flux_arrow!(ax, x, y_start, y_end, flux_obs, label, color; 
                         base_scale=300.0, labelside=:left, label_offset=(0.0, 0.0))
    
    # Width scaling
    width_obs = map(flux_obs) do f
        normalized = abs(f) / base_scale
        width_factor = normalized^0.7
        clamp(width_factor * 35, 3.0, 40.0)
    end
    
    direction = sign(y_end - y_start)
    
    # Smaller head dimensions (reduced scaling factors)
    head_width = map(width_obs) do w
        clamp(w * 0.018, 0.10, 0.30)  # Reduced from 0.03
    end
    
    head_height_obs = map(width_obs) do w
        clamp(w * 0.008, 0.08, 0.15) * direction  # Reduced from 0.012
    end
    
    # Shaft end position
    y_shaft_end = lift(head_height_obs) do hh
        y_end - hh
    end
    
    # Draw shaft with outline using two lines
    # Main colored shaft
    lines!(ax, [x, x], lift(y_shaft_end) do yse
               [y_start, yse]
           end, 
           linewidth = width_obs, color = color)
    
    # Black outline (slightly wider)
    outline_width = map(width_obs) do w
        w + 4  # 2 pixels wider on each side
    end
    
    lines!(ax, [x, x], lift(y_shaft_end) do yse
               [y_start, yse]
           end, 
           linewidth = outline_width, color = :black)
    
    # Re-draw colored shaft on top
    lines!(ax, [x, x], lift(y_shaft_end) do yse
               [y_start, yse]
           end, 
           linewidth = width_obs, color = color)
    
    # Arrow head
    arrow_head = lift(head_width, head_height_obs) do hw, hh
        [
            Point2f(x - hw, y_end - hh),
            Point2f(x + hw, y_end - hh),
            Point2f(x, y_end)
        ]
    end
    
    poly!(ax, arrow_head, color = color, strokecolor = :black, strokewidth = 2)
    
    # Label positioning
    label_x_base = labelside == :left ? (x - 0.35) : (x + 0.35)
    label_x = label_x_base + label_offset[1]
    label_y = (y_start + y_end) / 2 + label_offset[2]
    
    # Dynamic label with value
    label_text = map(flux_obs) do f
        val = round(Int, abs(f))
        "$(label)\n$(val) W/m²"
    end
    
    text!(ax, label_x, label_y, 
          text = label_text, 
          fontsize = 20, 
          align = (:center, :center),
          color = :black,
          font = :bold)
end

# Draw boxes
# Atmosphere box
poly!(ax, Point2f[(-3.8, -1.4), (3.8, -1.4), (3.8, 2.4), (-3.8, 2.4)],
      color = (:skyblue, 0.3), strokecolor = :steelblue, strokewidth = 3)

text!(ax, 0, 1.9, text = "ATMOSPHERE", fontsize = 30, align = (:center, :center), 
      font = :bold, color = :grey0)

# Surface box
poly!(ax, Point2f[(-3.8, -2.4), (3.8, -2.4), (3.8, -1.4), (-3.8, -1.4)],
      color = (:seagreen, 0.35), strokecolor = :darkgreen, strokewidth = 3)

text!(ax, 0, -1.8, text = "SURFACE", fontsize = 30, align = (:center, :center), 
      font = :bold, color = :grey0)

# Space indicator
#lines!(ax, [-3.0, 3.0], [2.3, 2.3], color = :black, linewidth = 2, linestyle = :dash)
#text!(ax, 2.5, 2.5, text = "Space", fontsize = 20, color = :gray4)

# Original cloud centers (right cloud)
cloud_centers = [
    (1.9, 1.2),
    (2.05, 1.18),
    (1.75, 1.18),
    (1.97, 1.25),
    (1.83, 1.25),
    (1.9, 1.33),
    (2.15, 1.16),
    (1.65, 1.16)
]

xs = first.(cloud_centers)
ys = last.(cloud_centers)

cloud_radii = [0.31, 0.28, 0.28, 0.26, 0.26, 0.25, 0.24, 0.24]  # 8 radii to match 8 centers
base_ms = cloud_radii .* 300

# Parameters for the second cloud (up & left, slight overlap)
dx = -0.60          # negative: left
dy = 0.22           # positive: up
scale_factor = 0.98 # slightly scale the radii (optional, tweak for overlap)

# Build second cloud centers from original (shifted)
second_centers = [(x + dx, y + dy) for (x, y) in cloud_centers]
xs2 = first.(second_centers)
ys2 = last.(second_centers)

cloud_radii2 = cloud_radii .* scale_factor
base_ms2 = cloud_radii2 .* 300

# Draw second cloud FIRST so it appears behind the original
# Outline (second cloud)
scatter!(ax,
    xs2, ys2,
    marker = :circle,
    markersize = base_ms2 .+ 11,   # outline thickness
    color = :black,
    strokewidth = 0
)

# Fill (second cloud)
scatter!(ax,
    xs2, ys2,
    marker = :circle,
    markersize = cloud_radii2 .* 300,
    color = :grey96,
    strokecolor = :transparent,
    strokewidth = 0
)

# Parameters for the third cloud 
dx3 = 0.35
dy3 = 0.30
scale_factor3 = 0.90

# Cloud 3 radii
radii_jitter = [1.00, 0.95, 0.95, 1.08, 1.08, 0.97, 0.96, 0.94]

third_centers = [(x + dx3, y + dy3) for (x, y) in cloud_centers]
xs3 = first.(third_centers)
ys3 = last.(third_centers)

cloud_radii3 = cloud_radii .* scale_factor3 .* radii_jitter
base_ms3 = cloud_radii3 .* 300

# Third cloud drawing
# Outline
scatter!(ax,
    xs3, ys3,
    marker = :circle,
    markersize = base_ms3 .+ 10,
    color = :black,
    strokewidth = 0
)

# Fill
scatter!(ax,
    xs3, ys3,
    marker = :circle,
    markersize = cloud_radii3 .* 300,
    color = :grey96,
    strokecolor = :transparent,
    strokewidth = 0
)


# Original cloud (at the front)
##Outline
scatter!(ax,
    xs, ys,
    marker = :circle,
    markersize = base_ms .+ 11,   # thickness of outline
    color = :black,
    strokewidth = 0
)

##Fill

scatter!(ax,
    xs, ys,
    marker = :circle,
    markersize = cloud_radii .* 300,  # pixel sizes
    color = :grey96,
    strokecolor = :transparent,
    strokewidth = 0
)



# Draw all flux arrows

## -------Shortwave fluxes-------

# Incoming solar 

# Then draw the arrow
draw_flux_arrow!(ax, -2.2, 2.9, 2.4, solar_constant, "Solar Constant", :khaki1, 
                base_scale=350, labelside=:left, label_offset=(-0.2, 0.4))
#draw_flux_arrow!(ax, -2.2, 2.6, 1.8, diagn.solarzenith, "Solar In", :khaki1, 
 #               base_scale=350, labelside=:left, label_offset=(-0.15, 0.2))

# Surface Shortwave up 
draw_flux_arrow!(ax, -1.9, -1.4, -0.4, SSRU_obs, "Surface SW
Up", :yellow, 
                base_scale=250, labelside=:left, label_offset=(0.9, 0.2))

# Outgoing shortwave to space 
draw_flux_arrow!(ax, -1.3, 2.4, 2.9, OSR_obs, "OSR", :goldenrod1, 
                base_scale=250, labelside=:right, label_offset=(0.3, 0.15))

# Shortwave absorbed at surface
draw_flux_arrow!(ax, -2.6, 0.5, -1.4, SSRD_obs, "Surface SW 
down", :gold1, 
                base_scale=280, labelside=:left, label_offset=(-0.3,0))


## -------Longwave fluxes-------
# Outgoing longwave to space (top, right)
draw_flux_arrow!(ax, 2.2, 2.4, 2.9, OLR_obs, "OLR", :darkred, 
                base_scale=300, labelside=:right, label_offset=(0.3, 0))

# Longwave down to surface
draw_flux_arrow!(ax, 1.9, 0.7, -1.4, SLRD_obs, "LW Down", :firebrick1, 
                base_scale=400, labelside=:right, label_offset=(0.09, 0))

# Longwave up from surface
draw_flux_arrow!(ax, 2.9, -1.4, 0.9, SLRU_obs, "LW Up", :red, 
                base_scale=450, labelside=:right, label_offset=(0.16, 0))

## Other Fluxes
# Surface to atmosphere: Sensible heat (short)
draw_flux_arrow!(ax, 0.4, -1.4, -0, SHF_obs, "Sensible", :deeppink1, 
                base_scale=120, labelside=:left, label_offset=(-0.2, 0))

# Surface to atmosphere: Latent heat (short)
draw_flux_arrow!(ax, 1.3, -1.4, 0, LHF_obs, "Latent", :slateblue, 
                base_scale=180, labelside=:left, label_offset=(-0.01, 0))


# Info panel 
info_grid = GridLayout(fig[1, 2], tellwidth = true)

Label(info_grid[1, 1], "Energy Balance", fontsize = 22, font = :bold, 
      halign = :center, color = :steelblue)

Label(info_grid[2, 1], "──────────────────────", fontsize = 15, halign = :center)

vals_text = map(current_point) do p
    """
    Surface Net: $(round(Int, p.surface_net)) W/m²
    
    Albedo: $(round(p.albedo, digits=3))
    
    Shortwave:
    • Surface Shortwave Down: $(round(Int, p.SSRD)) W/m²
    • OSR $(round(Int, p.OSR)) W/m²
    • Surface Shortwave Up: $(round(Int, p.SSRU)) W/m²
    • Net surface Shortwave: $(round(Int, p.SW_net_sfc)) W/m²
    
    Longwave:
    • OLR: $(round(Int, p.OLR)) W/m²
    • LW Up: $(round(Int, p.SLRU)) W/m²
    • LW Down: $(round(Int, p.SLRD)) W/m²
    • Net surface Longwave: $(round(Int, p.LW_net_sfc)) W/m²

    Other Fluxes:
    • LHF: $(round(Int, p.LHF)) W/m²
    • SHF: $(round(Int, p.SHF)) W/m²

    """
end

Label(info_grid[3, 1], vals_text, fontsize = 20, halign = :center, 
      valign = :top, tellheight = false)

time_text = map(current_point) do p
    "$(Dates.format(p.datetime, "yyyy-mm-dd HH:MM"))\nStep $(time_idx[])/$(nsteps)"
end

Label(info_grid[6, 1], time_text, fontsize = 20, halign = :center, color = :navyblue)

Label(info_grid[5, 1], "──────────────────────", fontsize = 15, halign = :center)
Label(info_grid[7, 1], "Legend", fontsize = 20, font = :bold, halign = :center)

legend_text = """
Yellow: Shortwave
Red: Longwave
Pink: Sensible heat
Blue: Latent heat

Arrow width shows flux magnitude
"""
Label(info_grid[8, 1], legend_text, fontsize =20, halign = :center)

# Controls 
control_grid = GridLayout(fig[2, 1:2])

slider = Slider(control_grid[1, 1], range = 1:nsteps, startvalue = 1)

is_playing = Observable(false)
play_label = Observable("Play")
play_button = Button(control_grid[1, 2], label = play_label, tellwidth = false)
reset_button = Button(control_grid[1, 3], label = "Reset", tellwidth = false)

# Sync slider and time_idx
on(slider.value) do v
    time_idx[] = Int(round(v))
end

on(time_idx) do i
    set_close_to!(slider, i)
end

# Animation control
animation_task = Ref{Union{Task, Nothing}}(nothing)

function stop_animation()
    is_playing[] = false
    if animation_task[] !== nothing
        try
            schedule(animation_task[], InterruptException(), error=true)
        catch
        end
        animation_task[] = nothing
    end
end

function start_animation()
    stop_animation()
    is_playing[] = true
    
    animation_task[] = @async begin
        try
            while is_playing[]
                if time_idx[] >= nsteps
                    time_idx[] = 1
                else
                    time_idx[] = time_idx[] + 1
                end
                sleep(0.06)
            end
        catch e
            if !(e isa InterruptException)
                @warn "Animation error" exception=e
            end
        end
    end
end

on(play_button.clicks) do _
    if is_playing[]
        stop_animation()
        play_label[] = "Play"
    else
        start_animation()
        play_label[] = "Pause"
    end
end

on(reset_button.clicks) do _
    stop_animation()
    play_label[] = "Play"
    time_idx[] = 1
end

# Auto-start animation
start_animation()
play_label[] = "Pause"

display(fig)

# Cleanup function
function cleanup()
    stop_animation()
end

cleanup (generic function with 1 method)